<a href="https://colab.research.google.com/github/johanhoffman/DD2365_FEniCSx/blob/main/Brinkman_NSE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# About the code

In [ ]:
# Copyright (C) 2019-2026 Johan Hoffman (jhoffman@kth.se)

# Setup environment

In [ ]:
# --- canonical: bootstrap v2 ---
import os as _os
_os.environ.setdefault("OMP_NUM_THREADS", "1")
_os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
_os.environ.setdefault("VECLIB_MAXIMUM_THREADS", "1")
_os.environ.setdefault("MKL_NUM_THREADS", "1")

import sys, os, subprocess

_on_colab = "google.colab" in sys.modules

if _on_colab:
    try:
        import gmsh
    except ImportError:
        subprocess.run(
            'wget -q "https://fem-on-colab.github.io/releases/gmsh-install.sh"'
            ' -O /tmp/gmsh-install.sh && bash /tmp/gmsh-install.sh',
            shell=True, check=True,
        )
    try:
        import dolfinx
    except ImportError:
        subprocess.run(
            'wget -q "https://fem-on-colab.github.io/releases/fenicsx-install-release-real.sh"'
            ' -O /tmp/fenicsx-install.sh && bash /tmp/fenicsx-install.sh',
            shell=True, check=True,
        )

import dolfinx
print("dolfinx version:", dolfinx.__version__)

if _on_colab:
    from google.colab import files
# --- end canonical: bootstrap ---

In [ ]:
# --- canonical: refine_cells v1 ---
import numpy as np
import dolfinx.mesh
from dolfinx.mesh import RefinementOption


def refine_cells(msh, predicate_or_mask):
    """Refine selected cells using the Plaza algorithm.

    Parameters
    ----------
    msh : dolfinx.mesh.Mesh
    predicate_or_mask : callable or array-like of bool
        Either a callable ``f(midpoints) -> bool array`` where
        ``midpoints`` has shape ``(ncells, gdim)``, or a boolean array
        of length ``num_local_cells`` that directly marks which cells
        to refine (useful when marks come from an existing DG0 field).

    Returns
    -------
    refined_msh : dolfinx.mesh.Mesh
    parent_cells : np.ndarray[np.int32]  shape (num_refined_cells,)
    parent_facets : np.ndarray[np.int8]  shape (num_refined_cells,)
        Local parent-facet index per refined cell, or -1 for interior.
    """
    tdim = msh.topology.dim
    num_cells = msh.topology.index_map(tdim).size_local

    if callable(predicate_or_mask):
        msh.topology.create_entities(1)
        msh.topology.create_connectivity(tdim, 0)
        midpoints = dolfinx.mesh.compute_midpoints(
            msh, tdim, np.arange(num_cells, dtype=np.int32)
        )
        marked = np.where(predicate_or_mask(midpoints))[0].astype(np.int32)
    else:
        mask = np.asarray(predicate_or_mask, dtype=bool)
        marked = np.where(mask[:num_cells])[0].astype(np.int32)

    if len(marked) == 0:
        n = msh.topology.index_map(tdim).size_local
        return msh, np.arange(n, dtype=np.int32), np.full(n, -1, dtype=np.int8)

    msh.topology.create_entities(1)
    msh.topology.create_connectivity(tdim, 1)
    edges = dolfinx.mesh.compute_incident_entities(msh.topology, marked, tdim, 1)
    edges = np.unique(edges).astype(np.int32)
    return dolfinx.mesh.refine(
        msh, edges, option=RefinementOption.parent_cell_and_facet
    )
# --- end canonical: refine_cells ---

In [ ]:
# --- canonical: tag_boundaries v1 ---
import numpy as np
from dolfinx.mesh import locate_entities_boundary, meshtags


def tag_boundaries(msh, L, H, eps=None):
    """Tag exterior boundary facets of a [0,L]×[0,H] rectangle with holes.

    Assigns integer tags to all exterior boundary facets:

        left=1  (x ≈ 0),   right=2 (x ≈ L),
        lower=3 (y ≈ 0),   upper=4 (y ≈ H),
        objects=5 (remaining exterior facets — circle boundaries).

    Corners are unambiguous: boundary facets are edges, not vertices, so a
    corner vertex is shared by one vertical and one horizontal edge — each
    edge belongs to exactly one group and no facet is tagged twice.

    Parameters
    ----------
    msh : dolfinx.mesh.Mesh
    L, H : float
        Rectangle dimensions.
    eps : float, optional
        Coordinate tolerance.  Default ``1e-6 * max(L, H)``.

    Returns
    -------
    dolfinx.mesh.MeshTags
        Must be recomputed whenever the mesh changes (e.g. after refinement).
    """
    if eps is None:
        eps = 1e-6 * max(L, H)

    fdim = msh.topology.dim - 1
    msh.topology.create_entities(fdim)
    msh.topology.create_connectivity(fdim, msh.topology.dim)

    left  = locate_entities_boundary(msh, fdim, lambda x: x[0] <= eps)
    right = locate_entities_boundary(msh, fdim, lambda x: x[0] >= L - eps)
    lower = locate_entities_boundary(msh, fdim, lambda x: x[1] <= eps)
    upper = locate_entities_boundary(msh, fdim, lambda x: x[1] >= H - eps)

    known = np.unique(np.concatenate([left, right, lower, upper]))
    all_bdry = locate_entities_boundary(
        msh, fdim, lambda x: np.ones(x.shape[1], dtype=bool)
    )
    objects = np.setdiff1d(all_bdry, known)

    indices = np.concatenate([left, right, lower, upper, objects]).astype(np.int32)
    values  = np.concatenate([
        np.full(len(left),    1, dtype=np.int32),
        np.full(len(right),   2, dtype=np.int32),
        np.full(len(lower),   3, dtype=np.int32),
        np.full(len(upper),   4, dtype=np.int32),
        np.full(len(objects), 5, dtype=np.int32),
    ])
    order = np.argsort(indices)
    return meshtags(msh, fdim, indices[order], values[order])
# --- end canonical: tag_boundaries ---

In [ ]:
# --- canonical: plot_helpers v3 ---
import numpy as np
import matplotlib.pyplot as plt
import basix.ufl as _bufl
from dolfinx.fem import functionspace, Function


def plot_mesh(msh, title="Mesh"):
    """Plot a 2-D triangular mesh using matplotlib triplot."""
    msh.topology.create_connectivity(msh.topology.dim, 0)
    x = msh.geometry.x
    gdm = msh.geometry.dofmaps[0]
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.triplot(x[:, 0], x[:, 1], gdm, linewidth=0.3, color="k")
    ax.set_aspect("equal")
    ax.set_title(title)
    plt.tight_layout()
    plt.show()


def _p1_scalar_space(msh):
    return functionspace(msh, ("Lagrange", 1))


def _scatter_p1_scalar(f1):
    """Scatter a P1 scalar Function values to geometry nodes.

    Returns (x, geometry_connectivity, node_values).
    """
    V = f1.function_space
    msh = V.mesh
    x = msh.geometry.x
    gdm = msh.geometry.dofmaps[0]
    ldm = V.dofmap.list
    vals = np.zeros(x.shape[0])
    vals[gdm.ravel()] = f1.x.array.real[ldm.ravel()]
    return x, gdm, vals


def plot_p1(u, title="Solution"):
    """Plot a P1 scalar Function using matplotlib tripcolor (Gouraud shading)."""
    plot_scalar(u, title=title)


def plot_scalar(f, title="Scalar field", ax=None):
    """Plot any scalar Lagrange Function via tripcolor (interpolates to P1 if needed).

    Parameters
    ----------
    f : dolfinx.fem.Function   scalar; any Lagrange degree
    title : str
    ax : matplotlib.axes.Axes or None
        If given, draw into this axes (no new figure, no plt.show()).
        If None (default), create a new figure and call plt.show().
    """
    V = f.function_space
    msh = V.mesh
    el = V.ufl_element()
    if el.degree == 1 and el.reference_value_shape == ():
        f1 = f
    else:
        f1 = Function(_p1_scalar_space(msh))
        f1.interpolate(f)
    x, gdm, vals = _scatter_p1_scalar(f1)
    _standalone = ax is None
    if _standalone:
        fig, ax = plt.subplots(figsize=(8, 3))
    tc = ax.tripcolor(x[:, 0], x[:, 1], gdm, vals, shading="gouraud")
    plt.colorbar(tc, ax=ax)
    ax.set_aspect("equal")
    ax.set_title(title)
    if _standalone:
        plt.tight_layout()
        plt.show()


def plot_vector(u, title="Vector field", quiver=True, quiver_stride=8, ax=None):
    """Plot a 2-D vector Lagrange Function as colour map of |u| + optional quiver.

    Parameters
    ----------
    u : dolfinx.fem.Function   value shape (2,); any Lagrange degree
    title : str
    quiver : bool              overlay subsampled arrows (default True)
    quiver_stride : int        take every N-th mesh node for arrows
    ax : matplotlib.axes.Axes or None
        If given, draw into this axes (no new figure, no plt.show()).
        If None (default), create a new figure and call plt.show().
    """
    msh = u.function_space.mesh
    gdim = msh.geometry.dim
    x = msh.geometry.x
    gdm = msh.geometry.dofmaps[0]
    npts = x.shape[0]

    # Interpolate to P1 vector so values align with geometry nodes
    V1v = functionspace(msh, _bufl.element("Lagrange", msh.basix_cell(), 1, shape=(gdim,)))
    u1v = Function(V1v)
    u1v.interpolate(u)

    # Scatter: for a block-size-2 P1 space, dofmap.list gives block indices
    ldm = V1v.dofmap.list      # (ncells, 3)  — block indices
    arr = u1v.x.array.real     # length npts * gdim, interleaved per block
    ux = np.zeros(npts)
    uy = np.zeros(npts)
    ux[gdm.ravel()] = arr[ldm.ravel() * gdim + 0]
    uy[gdm.ravel()] = arr[ldm.ravel() * gdim + 1]
    mag = np.sqrt(ux**2 + uy**2)

    _standalone = ax is None
    if _standalone:
        fig, ax = plt.subplots(figsize=(8, 3))
    tc = ax.tripcolor(x[:, 0], x[:, 1], gdm, mag, shading="gouraud", cmap="viridis")
    plt.colorbar(tc, ax=ax, label="|u|")
    if quiver:
        idx = np.arange(0, npts, quiver_stride)
        ax.quiver(x[idx, 0], x[idx, 1], ux[idx], uy[idx],
                  color="white", alpha=0.6, width=0.002, scale_units="xy", scale=2.0)
    ax.set_aspect("equal")
    ax.set_title(title)
    if _standalone:
        plt.tight_layout()
        plt.show()
# --- end canonical: plot_helpers ---

In [ ]:
# --- canonical: xdmf_series v1 ---
import sys
import pathlib
import basix.ufl as _bufl
from mpi4py import MPI
from dolfinx.io import XDMFFile
from dolfinx.fem import functionspace, Function


def _to_p1(f):
    """Interpolate f to P1 (scalar or vector) if needed."""
    V = f.function_space
    msh = V.mesh
    el = V.ufl_element()
    vshape = el.reference_value_shape
    if el.degree == 1:
        return f
    if vshape == ():
        V1 = functionspace(msh, ("Lagrange", 1))
    else:
        V1 = functionspace(msh, _bufl.element("Lagrange", msh.basix_cell(), 1, shape=vshape))
    f1 = Function(V1, name=f.name)
    f1.interpolate(f)
    return f1


class XDMFSeries:
    """Time-series XDMF writer.

    Opens one XDMFFile, writes the mesh once on the first write() call,
    then appends each time step.  Higher-degree functions are automatically
    interpolated to P1 before writing.

    Usage::

        xdmf = XDMFSeries("output/solution.xdmf")
        for t in ...:
            xdmf.write([u1, p1, w1], t)
        xdmf.close()          # or: `with XDMFSeries(...) as xdmf:`

    Parameters
    ----------
    path : str or Path
        Output ``.xdmf`` file path.  The ``.h5`` sidecar is written alongside.
    download : bool, optional
        If True *and* running on Colab, tar the pair and trigger a download
        when :meth:`close` is called.
    """

    def __init__(self, path, download=False):
        self._path = pathlib.Path(path)
        self._path.parent.mkdir(parents=True, exist_ok=True)
        self._download = download
        self._xdmf = XDMFFile(MPI.COMM_WORLD, str(self._path), "w")
        self._mesh_written = False

    def write(self, funcs, t):
        """Write *funcs* at time *t*."""
        funcs_p1 = [_to_p1(f) for f in funcs]
        if not self._mesh_written:
            self._xdmf.write_mesh(funcs_p1[0].function_space.mesh)
            self._mesh_written = True
        for f in funcs_p1:
            self._xdmf.write_function(f, t)

    def close(self):
        """Close the file and optionally trigger a Colab download."""
        self._xdmf.close()
        if self._download and "google.colab" in sys.modules:
            import tarfile
            from google.colab import files as colab_files
            tar_path = str(self._path.with_suffix(".tar.gz"))
            with tarfile.open(tar_path, "w:gz") as tar:
                tar.add(str(self._path), arcname=self._path.name)
                h5 = self._path.with_suffix(".h5")
                if h5.exists():
                    tar.add(str(h5), arcname=h5.name)
            colab_files.download(tar_path)

    def __enter__(self):
        return self

    def __exit__(self, *_):
        self.close()
# --- end canonical: xdmf_series ---

In [ ]:
# --- canonical: time_step v1 ---
import numpy as np


def time_step(msh, U_ref, C_CFL=0.5):
    """Return C_CFL * h_min / U_ref (CFL-based time step)."""
    tdim = msh.topology.dim
    n_cells = msh.topology.index_map(tdim).size_local
    h_min = float(msh.h(tdim, np.arange(n_cells, dtype=np.int32)).min())
    return C_CFL * h_min / U_ref
# --- end canonical: time_step ---

In [ ]:
import numpy as np
import time
import math
from mpi4py import MPI
from dolfinx.fem import (
    functionspace, Function, Constant, form,
)
from dolfinx.fem.petsc import (
    assemble_matrix, assemble_vector, apply_lifting, set_bc,
    create_matrix, create_vector,
)
from petsc4py import PETSc
import ufl
import basix.ufl as bufl
from dolfinx.mesh import create_rectangle, CellType, DiagonalType
import matplotlib.pyplot as plt

# Define domain and mesh

In [ ]:
from mpi4py import MPI
L = 4
H = 1
resolution = 16

msh = create_rectangle(
    MPI.COMM_WORLD,
    [[0.0, 0.0], [L, H]],
    [L * resolution, H * resolution],
    CellType.triangle,
    diagonal=DiagonalType.right,
)
facet_tags = tag_boundaries(msh, L, H)

# Keep refinement loop (no_levels=0 → no actual refinement)
no_levels = 0
for _ in range(no_levels):
    msh, _, _ = refine_cells(
        msh,
        lambda midpoints: np.hypot(midpoints[:, 0] - 0.5, midpoints[:, 1] - 0.5) < 1.0,
    )
    facet_tags = tag_boundaries(msh, L, H)

tdim = msh.topology.dim
fdim = tdim - 1

plot_mesh(msh)

# Define approximation spaces

In [ ]:
P1v = bufl.element("Lagrange", msh.basix_cell(), 1, shape=(2,))
P1s = bufl.element("Lagrange", msh.basix_cell(), 1)
DG0 = bufl.element("DG", msh.basix_cell(), 0)

V = functionspace(msh, P1v)
Q = functionspace(msh, P1s)
K = functionspace(msh, DG0)

u = ufl.TrialFunction(V); v = ufl.TestFunction(V)
p = ufl.TrialFunction(Q); q = ufl.TestFunction(Q)

V0, _ = V.sub(0).collapse()
V1, _ = V.sub(1).collapse()

# Define boundary conditions

In [ ]:
import math
from dolfinx.fem import locate_dofs_topological, dirichletbc

def _bc(space, value, tag, fdim=fdim):
    dofs = locate_dofs_topological(space, fdim, facet_tags.find(tag))
    return dirichletbc(PETSc.ScalarType(value), dofs, space)

bcu_in0  = _bc(V.sub(0), 1.0, 1)
bcu_in1  = _bc(V.sub(1), 0.0, 1)
bcu_upp0 = _bc(V.sub(0), 0.0, 4)
bcu_upp1 = _bc(V.sub(1), 0.0, 4)
bcu_low0 = _bc(V.sub(0), 0.0, 3)
bcu_low1 = _bc(V.sub(1), 0.0, 3)
bcu_obj0 = _bc(V.sub(0), 0.0, 5)
bcu_obj1 = _bc(V.sub(1), 0.0, 5)
bcp1     = _bc(Q,         0.0, 2)

bcu = [bcu_in0, bcu_in1, bcu_upp0, bcu_upp1, bcu_low0, bcu_low1, bcu_obj0, bcu_obj1]
bcp = [bcp1]

# pin placeholder (applied in time loop as: pin = 5.0 * abs(math.sin(t)))
pin = 5.0 * abs(math.sin(0.0))

# Define model parameters

In [ ]:
# DOF at cell centroid in both legacy (Expression) and dolfinx — interpolate evaluates lambda at centroid coordinates.
Kinv11 = Function(K, name="Kinv11")
Kinv11.interpolate(lambda x: 100.0 * np.abs(np.sin(2.0 * np.pi * (x[1] - 0.1 * x[0]))))
Kinv12 = Function(K, name="Kinv12")
Kinv12.x.array[:] = 0.0
Kinv21 = Function(K, name="Kinv21")
Kinv21.x.array[:] = 0.0
Kinv22 = Function(K, name="Kinv22")
Kinv22.interpolate(lambda x: 100.0 * np.abs(np.sin(2.0 * np.pi * (x[1] - 0.1 * x[0]))))

nu    = 1.0e-2
nueff = 1.0e-2

# Define method parameters

In [ ]:
# Define functions
u0 = Function(V, name="u0")
u1 = Function(V, name="u1")
p0 = Function(Q, name="p0")  # defined for legacy fidelity; not used in solve
p1 = Function(Q, name="p1")

# Set method parameters
num_nnlin_iter = 5

# Time step length
n_cells = msh.topology.index_map(tdim).size_local
dt = time_step(msh, 1.0)
dt_c = Constant(msh, PETSc.ScalarType(dt))

nu_c    = Constant(msh, PETSc.ScalarType(nu))
nueff_c = Constant(msh, PETSc.ScalarType(nueff))

ds = ufl.Measure("ds", domain=msh, subdomain_data=facet_tags)

# Simulation time and output frequency
T = 10
plot_freq = 10

# Define varitional problem

In [ ]:
# Define variational problems
h_c   = ufl.CellDiameter(msh)
u_mag = ufl.sqrt(ufl.dot(u1, u1))
d1    = 1.0 / ufl.sqrt((1.0 / dt_c) ** 2 + (u_mag / h_c) ** 2)
d2    = h_c * u_mag

um  = 0.5 * (u + u0)
um1 = 0.5 * (u1 + u0)

Fu = (ufl.inner((u - u0) / dt_c + ufl.dot(ufl.grad(um), um1), v) * ufl.dx
    - p1 * ufl.div(v) * ufl.dx
    + nueff_c * ufl.inner(ufl.grad(um), ufl.grad(v)) * ufl.dx
    + d1 * ufl.inner((u - u0) / dt_c + ufl.dot(ufl.grad(um), um1) + ufl.grad(p1),
                     ufl.dot(ufl.grad(v), um1)) * ufl.dx
    + d2 * ufl.div(um) * ufl.div(v) * ufl.dx
    + nu_c * (Kinv11 * ufl.inner(um[0], v[0]) * ufl.dx
            + Kinv12 * ufl.inner(um[0], v[1]) * ufl.dx
            + Kinv21 * ufl.inner(um[1], v[0]) * ufl.dx
            + Kinv22 * ufl.inner(um[1], v[1]) * ufl.dx))

Fp = (d1 * ufl.inner((u1 - u0) / dt_c + ufl.dot(ufl.grad(um1), um1) + ufl.grad(p),
                      ufl.grad(q)) * ufl.dx
    + ufl.div(um1) * q * ufl.dx)

a_u = form(ufl.lhs(Fu)); l_u = form(ufl.rhs(Fu))
a_p = form(ufl.lhs(Fp)); l_p = form(ufl.rhs(Fp))

A_u = create_matrix(a_u); b_u = create_vector(V)
A_p = create_matrix(a_p); b_p = create_vector(Q)

def _make_ksp_ilu(A):
    ksp = PETSc.KSP().create(msh.comm)
    ksp.setOperators(A); ksp.setType("bcgs"); ksp.getPC().setType("ilu")
    ksp.setTolerances(rtol=1e-6, atol=1e-14, max_it=500); ksp.setFromOptions()
    return ksp

def _make_ksp_amg(A):
    ksp = PETSc.KSP().create(msh.comm)
    ksp.setOperators(A); ksp.setType("bcgs")
    pc = ksp.getPC(); pc.setType("hypre"); pc.setHYPREType("boomeramg")
    ksp.setTolerances(rtol=1e-6, atol=1e-14, max_it=500); ksp.setFromOptions()
    return ksp

ksp_u = _make_ksp_ilu(A_u)
ksp_p = _make_ksp_amg(A_p)

# Initialise BCs on u0 and u1
set_bc(u0.x.petsc_vec, bcu); u0.x.scatter_forward()
set_bc(u1.x.petsc_vec, bcu); u1.x.scatter_forward()

# Set I/O variables

In [ ]:
xdmf_u = XDMFSeries("results-BNS/u.xdmf")
xdmf_p = XDMFSeries("results-BNS/p.xdmf")

plot_time = 0

# Time stepping algorithm

In [ ]:
t_wall0 = time.time()
t = dt
step = 0
while t < T + 1e-10:

    # pin = 5.0 * abs(math.sin(t))  # not applied; bcp0 excluded from bcp as in legacy

    # Solve non-linear problem
    k = 0
    while k < num_nnlin_iter:

        # Assemble momentum matrix and vector
        A_u.zeroEntries()
        assemble_matrix(A_u, a_u, bcs=bcu); A_u.assemble()
        with b_u.localForm() as loc: loc.set(0.0)
        assemble_vector(b_u, l_u)
        apply_lifting(b_u, [a_u], bcs=[bcu])
        b_u.ghostUpdate(addv=PETSc.InsertMode.ADD, mode=PETSc.ScatterMode.REVERSE)
        set_bc(b_u, bcu)
        ksp_u.setOperators(A_u, A_u); ksp_u.solve(b_u, u1.x.petsc_vec)
        u1.x.scatter_forward()
        set_bc(u1.x.petsc_vec, bcu); u1.x.scatter_forward()

        # Assemble pressure matrix and vector
        A_p.zeroEntries()
        assemble_matrix(A_p, a_p, bcs=bcp); A_p.assemble()
        with b_p.localForm() as loc: loc.set(0.0)
        assemble_vector(b_p, l_p)
        apply_lifting(b_p, [a_p], bcs=[bcp])
        b_p.ghostUpdate(addv=PETSc.InsertMode.ADD, mode=PETSc.ScatterMode.REVERSE)
        set_bc(b_p, bcp)
        ksp_p.setOperators(A_p, A_p); ksp_p.solve(b_p, p1.x.petsc_vec)
        p1.x.scatter_forward()
        set_bc(p1.x.petsc_vec, bcp); p1.x.scatter_forward()

        k += 1

    if t > plot_time:

        s = "Time t = " + repr(t)
        print(s)

        xdmf_u.write([u1], t)
        xdmf_p.write([p1], t)

        plot_vector(u1, title="Velocity")
        plot_scalar(p1, title="Pressure")

        plot_time += T / plot_freq

        plt.show()

    u0.x.array[:] = u1.x.array
    t += dt
    step += 1

xdmf_u.close()
xdmf_p.close()

In [ ]:
from dolfinx.fem import assemble_scalar as _ascalar
norm_u1 = float(msh.comm.allreduce(
    _ascalar(form(ufl.inner(u1, u1) * ufl.dx)), op=MPI.SUM
)) ** 0.5
norm_p1 = float(msh.comm.allreduce(
    _ascalar(form(p1 * p1 * ufl.dx)), op=MPI.SUM
)) ** 0.5
print(f"||u1|| = {norm_u1:.6f}")
print(f"||p1|| = {norm_p1:.6f}")
print(f"cells  = {n_cells}")
print(f"dt     = {dt:.6f}")
print(f"steps  = {step}")
print(f"QOI norm_u = {norm_u1:.6g}")
print(f"QOI norm_p = {norm_p1:.6g}")